## Exploratory Data Analysis

### Import Libraries

In [1]:
import sys
import polars as pl
import polars.selectors as pol_sel

### Show Python & Library Versions

In [2]:
l = 8
r = 12

print("Python".rjust(l), ":", sys.version[0:6].ljust(r))
print("Polars".rjust(l), ":", pl.__version__.ljust(r))

  Python : 3.11.4      
  Polars : 1.12.0      


### Load CSV File into Polars DataFrame

In [3]:
df = pl.read_csv(
    "data/geolocation_dataset.csv", 
    ignore_errors=True, 
    truncate_ragged_lines=True
    )

df

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
i64,f64,f64,str,str
1037,-23.545621,-46.639292,"""sao paulo""","""SP"""
1046,-23.546081,-46.64482,"""sao paulo""","""SP"""
1046,-23.546129,-46.642951,"""sao paulo""","""SP"""
1041,-23.544392,-46.639499,"""sao paulo""","""SP"""
1035,-23.541578,-46.641607,"""sao paulo""","""SP"""
…,…,…,…,…
99950,-28.068639,-52.010705,"""tapejara""","""RS"""
99900,-27.877125,-52.224882,"""getulio vargas""","""RS"""
99950,-28.071855,-52.014716,"""tapejara""","""RS"""


### Retrieve Number of Nulls in Each Feature

In [4]:
def count_nulls(df: pl.DataFrame) -> pl.DataFrame:
    return pl.DataFrame({
        "feature": df.columns,
        "null_count": df.null_count().row(0)
    })

pl.Config.set_tbl_rows(35)

null_counts = count_nulls(df)
null_counts

feature,null_count
str,i64
"""geolocation_zip_code_prefix""",0
"""geolocation_lat""",0
"""geolocation_lng""",0
"""geolocation_city""",0
"""geolocation_state""",0


### Retrieve Basic Information About DataFrame

In [5]:
def print_schema(df: pl.DataFrame):
    print(f"{'Column':<30} | {'Data Type'}")
    print("-" * 60)
    for name, dtype in zip(df.columns, df.dtypes):
        print(f"{name:<30} | {dtype}")

print_schema(df)

Column                         | Data Type
------------------------------------------------------------
geolocation_zip_code_prefix    | Int64
geolocation_lat                | Float64
geolocation_lng                | Float64
geolocation_city               | String
geolocation_state              | String


### Display Summary Statistics for All Columns

In [6]:
summary = df.describe()
print(summary)

shape: (9, 6)
┌────────────┬─────────────────┬────────────────┬────────────────┬────────────────┬────────────────┐
│ statistic  ┆ geolocation_zip ┆ geolocation_la ┆ geolocation_ln ┆ geolocation_ci ┆ geolocation_st │
│ ---        ┆ _code_prefix    ┆ t              ┆ g              ┆ ty             ┆ ate            │
│ str        ┆ ---             ┆ ---            ┆ ---            ┆ ---            ┆ ---            │
│            ┆ f64             ┆ f64            ┆ f64            ┆ str            ┆ str            │
╞════════════╪═════════════════╪════════════════╪════════════════╪════════════════╪════════════════╡
│ count      ┆ 1.000163e6      ┆ 1.000163e6     ┆ 1.000163e6     ┆ 1000163        ┆ 1000163        │
│ null_count ┆ 0.0             ┆ 0.0            ┆ 0.0            ┆ 0              ┆ 0              │
│ mean       ┆ 36574.166466    ┆ -21.176153     ┆ -46.390541     ┆ null           ┆ null           │
│ std        ┆ 30549.33571     ┆ 5.715866       ┆ 4.269748       ┆ null      

### Find Longest Text Length in Each Column

In [7]:
# Create an empty list to store max lengths for each string column
longest_text_lengths = []

# Loop through the columns to check for string columns
string_columns = [col for col in df.columns if df[col].dtype == pl.Utf8]

max_lengths = {}
for col in string_columns:
    max_length = df.select(pl.col(col).str.len_chars().max()).to_numpy()[0, 0]
    max_lengths[col] = max_length

df_max_lengths = pl.DataFrame(max_lengths)

df_max_lengths

geolocation_city,geolocation_state
u32,u32
38,2


### Retrieve Data Types of All Columns

In [8]:
print("Column data types:\n", df.dtypes)

Column data types:
 [Int64, Float64, Float64, String, String]


### Count Unique Values in Each Column

In [9]:
all_columns = [col for col in df.columns]

for col in all_columns:
    unique_counts = df[col].n_unique()
    print(f"Unique values in {col} :".rjust(48), f"{unique_counts}".ljust(6))

  Unique values in geolocation_zip_code_prefix : 19015 
              Unique values in geolocation_lat : 717372
              Unique values in geolocation_lng : 717615
             Unique values in geolocation_city : 8011  
            Unique values in geolocation_state : 27    


### Check Distribution of Numerical Columns

In [10]:
numerical_cols = [item for item in all_columns if item not in string_columns]
numerical_cols = [item for item in numerical_cols if item not in ['ID']]

numerical_cols

for col in numerical_cols:
    distribution = df.select(col).describe()
    print(col)
    print(distribution, '\n\n')

geolocation_zip_code_prefix
shape: (9, 2)
┌────────────┬─────────────────────────────┐
│ statistic  ┆ geolocation_zip_code_prefix │
│ ---        ┆ ---                         │
│ str        ┆ f64                         │
╞════════════╪═════════════════════════════╡
│ count      ┆ 1.000163e6                  │
│ null_count ┆ 0.0                         │
│ mean       ┆ 36574.166466                │
│ std        ┆ 30549.33571                 │
│ min        ┆ 1001.0                      │
│ 25%        ┆ 11075.0                     │
│ 50%        ┆ 26530.0                     │
│ 75%        ┆ 63504.0                     │
│ max        ┆ 99990.0                     │
└────────────┴─────────────────────────────┘ 


geolocation_lat
shape: (9, 2)
┌────────────┬─────────────────┐
│ statistic  ┆ geolocation_lat │
│ ---        ┆ ---             │
│ str        ┆ f64             │
╞════════════╪═════════════════╡
│ count      ┆ 1.000163e6      │
│ null_count ┆ 0.0             │
│ mean       ┆ -21.

### List Unique Values For Certain Features

In [11]:
def list_unique_values_under_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count < threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_under_threshold(df)

Column: geolocation_state (27 unique values)
shape: (27,)
Series: 'geolocation_state' [str]
[
	"AC"
	"AL"
	"AM"
	"AP"
	"BA"
	"CE"
	"DF"
	"ES"
	"GO"
	"MA"
	"MG"
	"MS"
	"MT"
	"PA"
	"PB"
	"PE"
	"PI"
	"PR"
	"RJ"
	"RN"
	"RO"
	"RR"
	"RS"
	"SC"
	"SE"
	"SP"
	"TO"
]
--------------------------------------------------


In [12]:
def list_unique_values_over_threshold(df: pl.DataFrame, threshold: int = 5000):
    for col in df.columns:
        unique_count = df.select(pl.col(col).n_unique()).item()
        if unique_count > threshold:
            unique_vals = df.select(pl.col(col).unique().sort()).to_series()
            print(f"Column: {col} ({unique_count} unique values)")
            print(unique_vals)
            print("-" * 50)

# Apply to your DataFrame
list_unique_values_over_threshold(df)

Column: geolocation_zip_code_prefix (19015 unique values)
shape: (19_015,)
Series: 'geolocation_zip_code_prefix' [i64]
[
	1001
	1002
	1003
	1004
	1005
	1006
	1007
	1008
	1009
	1010
	1011
	1012
	1013
	1014
	1015
	1016
	1017
	1018
	…
	99880
	99890
	99895
	99900
	99910
	99920
	99925
	99930
	99940
	99950
	99952
	99955
	99960
	99965
	99970
	99980
	99990
]
--------------------------------------------------
Column: geolocation_lat (717372 unique values)
shape: (717_372,)
Series: 'geolocation_lat' [f64]
[
	-36.605374
	-36.603837
	-34.6224
	-34.586422
	-33.692616
	-33.692504
	-33.692491
	-33.692291
	-33.692255
	-33.692196
	-33.692177
	-33.692136
	-33.692047
	-33.691699
	-33.691673
	-33.691643
	-33.691608
	-33.69155
	…
	38.268205
	38.323939
	38.381672
	38.713128
	38.816816
	38.991963
	39.057629
	41.146203
	41.385328
	41.614052
	42.166805
	42.167251
	42.184003
	42.428884
	42.439286
	43.684961
	45.065933
]
--------------------------------------------------
Column: geolocation_lng (717615 unique va

In [13]:
# List the columns to check
cols = [
    "iowa_zip_code_tabulation_areas",
    "iowa_watershed_sub_basins",
    "iowa_watersheds",
    "county_boundaries_of_iowa"
]

# Create a boolean mask for each column being null, then sum the row-wise nulls
null_mask = df.select([pl.col(c).is_null().alias(c) for c in cols])

# Sum across the selected columns to see if all are null in the same row
same_null_rows = null_mask.select(
    (pl.sum_horizontal(null_mask.columns) == len(cols)).alias("all_null")
)

# Count how many rows have all those columns null at the same time
count_all_null = same_null_rows.filter(pl.col("all_null")).height

# Display the result
if count_all_null == 2506680:
    print(f"The nulls are all in the same Rows as: Rows where all {len(cols)} columns are null: {count_all_null}")

ColumnNotFoundError: iowa_zip_code_tabulation_areas

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'select' <---
DF ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng", "geolocation_city"]; PROJECT */5 COLUMNS; SELECTION: None

### How Many Records Remain IF I Remove Records With Any Nulls In It

In [ ]:
def drop_rows_with_any_nulls(df: pl.DataFrame) -> pl.DataFrame:
    """
    Removes all rows from a Polars DataFrame that contain any null values.
    """
    return df.drop_nulls()


drop_rows_with_any_nulls(df)

### Iterate Through Each File & Provide Length of Longest String For Each Feature

In [3]:
import polars as pl
import os

# Folder containing the CSV files
folder_path = "data"

# Iterate over all CSV files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        print(f"\n{filename}")
        
        # Read CSV into a Polars DataFrame
        df = pl.read_csv(file_path)
        
        # For each column, compute the max string length
        for col in df.columns:
            # Cast the column to string, then get the length of each value
            lengths = df[col].cast(pl.Utf8).str.len_chars()
            max_length = lengths.max()
            print(f"  '{col}': {max_length}")



order_payments_dataset.csv
  'order_id': 32
  'payment_sequential': 2
  'payment_type': 11
  'payment_installments': 2
  'payment_value': 8

product_category_name_translation.csv
  'product_category_name': 46
  'product_category_name_english': 39

orders_dataset.csv
  'order_id': 32
  'customer_id': 32
  'order_status': 11
  'order_purchase_timestamp': 19
  'order_approved_at': 19
  'order_delivered_carrier_date': 19
  'order_delivered_customer_date': 19
  'order_estimated_delivery_date': 19

order_items_dataset.csv
  'order_id': 32
  'order_item_id': 2
  'product_id': 32
  'seller_id': 32
  'shipping_limit_date': 19
  'price': 7
  'freight_value': 6

order_reviews_dataset.csv
  'review_id': 32
  'order_id': 32
  'review_score': 1
  'review_comment_title': 26
  'review_comment_message': 208
  'review_creation_date': 19
  'review_answer_timestamp': 19

sellers_dataset.csv
  'seller_id': 32
  'seller_zip_code_prefix': 5
  'seller_city': 40
  'seller_state': 2

geolocation_dataset.csv
  

In [5]:
import polars as pl
import os

# Folder containing the CSV files
folder_path = "data"
output_folder = "EDA_Results"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(folder_path, filename)
        output_path = os.path.join(output_folder, f"{filename}.txt")

        df = pl.read_csv(file_path)
        output_lines = [f"File: {filename}\n"]

        for col in df.columns:
            col_utf8 = df[col].cast(pl.Utf8)
            col_len = col_utf8.str.len_chars().max()
            null_count = df[col].null_count()
            nunique = df[col].n_unique()
            output_lines.append(f"\nColumn: '{col}'")
            output_lines.append(f"  Max String Length: {col_len}")
            output_lines.append(f"  Null Count: {null_count}")
            output_lines.append(f"  Unique Values: {nunique}")

            # Fetch sorted unique values
            try:
                sorted_unique = df[col].unique().sort()
            except:
                # fallback for unorderable types
                sorted_unique = df[col].unique()

            if nunique <= 500:
                output_lines.append(f"  Unique Values List ({nunique}):")
                for val in sorted_unique.to_list():
                    output_lines.append(f"    {val}")
            else:
                values = sorted_unique.to_list()
                output_lines.append(f"  First 12 of {nunique} unique values:")
                for val in values[:12]:
                    output_lines.append(f"    {val}")
                output_lines.append(f"  Last 12 of {nunique} unique values:")
                for val in values[-12:]:
                    output_lines.append(f"    {val}")

        # Write to output file
        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n".join(output_lines))

        print(f"✓ Analyzed {filename} → {output_path}")


✓ Analyzed order_payments_dataset.csv → EDA_Results/order_payments_dataset.csv.txt
✓ Analyzed product_category_name_translation.csv → EDA_Results/product_category_name_translation.csv.txt
✓ Analyzed orders_dataset.csv → EDA_Results/orders_dataset.csv.txt
✓ Analyzed order_items_dataset.csv → EDA_Results/order_items_dataset.csv.txt
✓ Analyzed order_reviews_dataset.csv → EDA_Results/order_reviews_dataset.csv.txt
✓ Analyzed sellers_dataset.csv → EDA_Results/sellers_dataset.csv.txt
✓ Analyzed geolocation_dataset.csv → EDA_Results/geolocation_dataset.csv.txt
✓ Analyzed products_dataset.csv → EDA_Results/products_dataset.csv.txt
✓ Analyzed customers_dataset.csv → EDA_Results/customers_dataset.csv.txt
